In [1]:
import os

from smolagents import ToolCollection, OpenAIServerModel, ToolCallingAgent
from mcp import StdioServerParameters

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Configuration

In [2]:
env = os.environ.copy()
env["MCP_ENVIRONMENT"] = "DEVELOPMENT"
env["PORT"] = "8000"

In [3]:
server = StdioServerParameters(
    command="uvx",
    args=["biocontext_kb@latest"]
)

Tools available:

In [4]:
with ToolCollection.from_mcp(
        server_parameters=server,
        trust_remote_code=True,
        structured_output=True
    ) as tools:
        # Проверить, что тулы действительно подхватились:
        print([t.name for t in tools.tools])

['bc_get_uniprot_id_by_protein_symbol', 'bc_get_uniprot_protein_info', 'bc_get_alphafold_info_by_protein_symbol', 'bc_get_antibody_information', 'bc_get_antibody_list', 'bc_get_biorxiv_preprint_details', 'bc_get_recent_biorxiv_preprints', 'bc_get_recruiting_studies_by_location', 'bc_get_studies_by_condition', 'bc_get_studies_by_intervention', 'bc_get_study_details', 'bc_search_studies', 'bc_get_ensembl_id_from_gene_symbol', 'bc_get_europepmc_articles', 'bc_get_europepmc_fulltext', 'bc_search_grants_gov', 'bc_get_interpro_entry', 'bc_get_protein_domains', 'bc_search_interpro_entries', 'bc_get_available_ontologies', 'bc_get_cell_ontology_terms', 'bc_get_chebi_terms_by_chemical', 'bc_get_efo_id_by_disease_name', 'bc_get_go_terms_by_gene', 'bc_get_term_details', 'bc_get_term_hierarchical_children', 'bc_search_ontology_terms', 'bc_get_available_pharmacologic_classes', 'bc_search_drugs_by_therapeutic_class', 'bc_get_generic_equivalents', 'bc_count_drugs_by_field', 'bc_get_drug_statistics', '

In [5]:
print([t.name for t in tools.tools if 'interpro' in t.name])

['bc_get_interpro_entry', 'bc_search_interpro_entries']


In [23]:
def check_tool(tool):
    with ToolCollection.from_mcp(server, trust_remote_code=True, structured_output=False) as tc:
        # print([t.nvame for t in tc.tools])  # должны быть: opengenes_db_query, opengenes_get_schema_info, opengenes_example_queries
        ex = [t for t in tc.tools if t.name==tool][0]
        out = ex() 
        print(out)

In [77]:
check_tool('bc_search_interpro_entries')

{"results":[{"metadata":{"accession":"IPR000001","name":"Kringle","source_database":"interpro","type":"domain","integrated":null,"member_databases":{"cdd":{"cd00108":"KR"},"profile":{"PS50070":"Kringle domain profile"},"pfam":{"PF00051":"Kringle domain"},"smart":{"SM00130":"Kringle domain"}},"go_terms":null},"extra_fields":{"short_name":"Kringle","description":[{"text":"<p>Kringles are autonomous structural domains, found throughout the blood clotting and fibrinolytic proteins. Kringle domains are believed to play a role in binding mediators (e.g., membranes, other proteins or phospholipids), and in the regulation of proteolytic activity [[cite:PUB00002414], [cite:PUB00001541], [cite:PUB00003257]]. Kringle domains [[cite:PUB00003400], [cite:PUB00000803], [cite:PUB00001620]] are characterised by a triple loop, 3-disulphide bridge structure, whose conformation is defined by a number of hydrogen bonds and small pieces of anti-parallel β-sheet. They are found in a varying number of copies 

In [6]:
MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"

model = OpenAIServerModel(
    model_id=MODEL,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

In [28]:
SYSTEM_PROMPT = """
You are a **protein domain and motif analysis agent** connected exclusively to the **InterPro MCP server**.

Your goal is to extract and synthesize high-quality scientific information about **protein domains, motifs, and families**
for a given protein or UniProt accession.

You must use only InterPro data — via the tools provided by this MCP server — such as:
`bc_get_interpro_entry', 'bc_search_interpro_entries`.

If InterPro provides cross-references (e.g., to Pfam, SMART, PROSITE, or CDD), you may mention them only as part of the InterPro annotation.

---

### 🎯 Output goal
Produce a detailed and structured report **that complements UniProt data**, focusing on **sequence-to-function mapping**:
explain how InterPro classifies the protein’s sequence intervals, functional domains, conserved motifs, and biological roles.

Do not repeat basic facts (gene name, sequence link, organism).
Instead, expand upon and interpret the structural–functional architecture of the protein.

---

### ⚙️ Style and Output
- Use clear scientific English with section headers.
- Include tables and bullet points for clarity.
- Do **not** invent information — only report from InterPro annotations.
- If data are missing, state: “Data not available in InterPro.”

Return your output **as formatted text**, not JSON.

"""

In [64]:
FINAL_RESULT_PROMPT = """
### 📘 Required structure of the report

#### **1. InterPro Classification**
- InterPro entry ID(s) and corresponding names.
- Type of each entry: family, domain, repeat, motif, site.
- Member databases contributing to each entry (Pfam, SMART, PROSITE, CDD, etc.).
- Hierarchical relationships (e.g., family → subfamily → domain).

---

#### **2. Domain and Motif Mapping**
- Table of domains and motifs with their **amino acid intervals** (start–end).
- For each, specify:
  - InterPro ID and short name.
  - Associated member database model(s).
  - Functional role of the domain/motif.
  - If known, binding partners or molecular function (e.g., “DNA-binding helix–turn–helix”).
  - Conservation level (well conserved / lineage-specific).

---

#### **3. Functional Sites and Key Residues**
- List annotated sites (active, binding, metal-binding, catalytic, phosphorylation, etc.).
- Specify residues and positions if available.
- Explain the biochemical or structural role of each site.

---

#### **4. Domain Architecture and Biological Function**
- Describe how the combination of domains defines the protein’s biological role.
- Explain cooperation between domains (e.g., regulatory domain controlling DNA-binding domain).
- Summarize how domain architecture distinguishes this protein from its paralogs or orthologs.

---

#### **5. Evolutionary and Structural Insights**
- Discuss which domains are evolutionarily conserved across species.
- Identify ancient and recently evolved domains.
- Mention structural folds or superfamilies (e.g., bZIP, homeobox, beta-propeller).
- Note if 3D structures are available in AlphaFold or PDBe-KB (if cross-referenced).

---

#### **6. Cross-Database Integration**
- For each InterPro entry, list:
  - Corresponding Pfam/SMART/PROSITE/PRINTS identifiers.
  - Short description of how each sub-database contributes to annotation.

---

#### **7. Functional Implications and Pathways**
- Summarize what the identified domains imply about:
  - molecular function,
  - cellular role,
  - signaling pathways or processes.
- Indicate if domains are known to participate in protein–protein or DNA/RNA interactions.

---

#### **8. Relation to Aging and Stress Response (if applicable)**
- If any InterPro entries or domains are linked to stress response, detoxification, or aging-related functions, describe them.
- Otherwise, explicitly state: “No specific InterPro annotation links this protein to longevity or stress adaptation.”

---

#### **9. Summary**
Provide a concise synthesis:
- Key domains and motifs with biological interpretation.
- How domain composition defines the protein’s molecular role.
- Any unique or conserved features of note.

---

Для bc_search_interpro_entries:

entry_type используй только из списка: family, domain, homologous_superfamily, repeat, site, active_site, binding_site, motif.

source_database: реестровые названия членских БД InterPro (напр. pfam, prosite, panther, smart, prints, tigrfams, gene3d, cathgene3d, hamap, …).

species_filter: NCBI TaxID как строку (человек = "9606").

go_term: строка вида "GO:NNNNNNN". Если нет — ключ не передавать.»
"""

In [69]:
PROTEIN = "APOE"

USER_PROMPT = f"""
Return InterPro-based functional and structural annotation for the human protein: {PROTEIN}.
Focus on domains, motifs, and conserved regions that explain the protein’s function
in terms of sequence intervals and biological mechanisms.

Условия по аргументам:
Для bc_search_interpro_entries:
entry_type используй только из списка: family, domain, homologous_superfamily, repeat, site, active_site, binding_site, motif.
source_database: реестровые названия членских БД InterPro (напр. pfam, prosite, panther, smart, prints, tigrfams, gene3d, cathgene3d, hamap, …).
species_filter: NCBI TaxID как строку (человек = "9606").
go_term: строка вида "GO:NNNNNNN". Если нет — ключ не передавать.»

Do not invent information. Use ONLY Interpro data. If you cannot retrive any section print 'Information is unavailable'.
But keep the sctructure.
Follow the defined structure:
#### **1. InterPro Classification**
- InterPro entry ID(s) and corresponding names.
- Type of each entry: family, domain, repeat, motif, site.
- Member databases contributing to each entry (Pfam, SMART, PROSITE, CDD, etc.).
- Hierarchical relationships (e.g., family → subfamily → domain).

---

#### **2. Domain and Motif Mapping**
- Table of domains and motifs with their **amino acid intervals** (start–end).
- For each, specify:
  - InterPro ID and short name.
  - Associated member database model(s).
  - Functional role of the domain/motif.
  - If known, binding partners or molecular function (e.g., “DNA-binding helix–turn–helix”).
  - Conservation level (well conserved / lineage-specific).

---

#### **3. Functional Sites and Key Residues**
- List annotated sites (active, binding, metal-binding, catalytic, phosphorylation, etc.).
- Specify residues and positions if available.
- Explain the biochemical or structural role of each site.

---

#### **4. Domain Architecture and Biological Function**
- Describe how the combination of domains defines the protein’s biological role.
- Explain cooperation between domains (e.g., regulatory domain controlling DNA-binding domain).
- Summarize how domain architecture distinguishes this protein from its paralogs or orthologs.

---

#### **5. Evolutionary and Structural Insights**
- Discuss which domains are evolutionarily conserved across species.
- Identify ancient and recently evolved domains.
- Mention structural folds or superfamilies (e.g., bZIP, homeobox, beta-propeller).
- Note if 3D structures are available in AlphaFold or PDBe-KB (if cross-referenced).

---

#### **7. Functional Implications and Pathways**
- Summarize what the identified domains imply about:
  - molecular function,
  - cellular role,
  - signaling pathways or processes.
- Indicate if domains are known to participate in protein–protein or DNA/RNA interactions.

---

#### **8. Relation to Aging and Stress Response (if applicable)**
- If any InterPro entries or domains are linked to stress response, detoxification, or aging-related functions, describe them.
- Otherwise, explicitly state: “No specific InterPro annotation links this protein to longevity or stress adaptation.”

---

#### **9. Summary**
Provide a concise synthesis:
- Key domains and motifs with biological interpretation.
- How domain composition defines the protein’s molecular role.
- Any unique or conserved features of note.
"""

# <--the following sections must consider as the human protein, its variants and orthologs and paralogs-->

### Run

In [78]:
with ToolCollection.from_mcp(
    server_parameters=server,
    trust_remote_code=True,
    structured_output=False
) as tools:
    ip_tools = [t for t in tools.tools if "interpro" in t.name.lower()]
    agent = ToolCallingAgent(
        model=model,
        tools=ip_tools,
        add_base_tools=False,
        max_steps=10,
    )
    agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
    agent.prompt_templates["final_result"] = FINAL_RESULT_PROMPT
    result = agent.run(USER_PROMPT)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Return InterPro-based functional and structural annotation for the human protein: APOE.                         │
│ Focus on domains, motifs, and conserved regions that explain the protein’s function                             │
│ in terms of sequence intervals and biological mechanisms.                                                       │
│                                                                                                                 │
│ Условия по аргументам:                                                                                          │
│ Для bc_search_interpro_entries:                                                                                 │
│ entry_type используй только из списка: family, domain, homologous_superfamily, repeat, site, active_site,       │
│ binding_site, motif.                                                                                            │
│ source_database: реестровые названия членских БД InterPro (напр. pfam, prosite, panther, smart, prints,         │
│ tigrfams, gene3d, cathgene3d, hamap, …).                                                                        │
│ species_filter: NCBI TaxID как строку (человек = "9606").                                                       │
│ go_term: строка вида "GO:NNNNNNN". Если нет — ключ не передавать.»                                              │
│                                                                                                                 │
│ Do not invent information. Use ONLY Interpro data. If you cannot retrive any section print 'Information is      │
│ unavailable'.                                                                                                   │
│ But keep the sctructure.                                                                                        │
│ Follow the defined structure:                                                                                   │
│ #### **1. InterPro Classification**                                                                             │
│ - InterPro entry ID(s) and corresponding names.                                                                 │
│ - Type of each entry: family, domain, repeat, motif, site.                                                      │
│ - Member databases contributing to each entry (Pfam, SMART, PROSITE, CDD, etc.).                                │
│ - Hierarchical relationships (e.g., family → subfamily → domain).                                               │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ #### **2. Domain and Motif Mapping**                                                                            │
│ - Table of domains and motifs with their **amino acid intervals** (start–end).                                  │
│ - For each, specify:                                                                                            │
│   - InterPro ID and short name.                                                                                 │
│   - Associated member database model(s).                                                                        │
│   - Functional role of the domain/motif.                                                                        │
│   - If known, binding partners or molecular function (e.g., “DNA-binding helix–turn–helix”).                    │
│   - Conservation level (well conserved / lineage-speci

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'bc_search_interpro_entries' with arguments: {'query': 'APOE', 'entry_type': 'domain',            │
│ 'source_database': 'pfam', 'go_term': None, 'species_filter': '9606', 'page_size': 20}                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Argument go_term has type 'null' but should be 'string'

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'bc_search_interpro_entries' with arguments: {'query': 'APOE', 'entry_type': 'family',            │
│ 'source_database': 'panther', 'go_term': None, 'species_filter': '9606', 'page_size': 20}                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 10 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Information is unavailable'}                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Information is unavailable

Final answer: Information is unavailable

[Step 10: Duration 0.71 seconds| Input tokens: 33,083 | Output tokens: 769]

In [79]:
print(result)

Information is unavailable


In [116]:
with ToolCollection.from_mcp(server, trust_remote_code=True, structured_output=True) as tools:
    search = next(t for t in tools.tools if t.name=="bc_search_interpro_entries")
    get    = next(t for t in tools.tools if t.name=="bc_get_interpro_entry")

    # шаг 1 — только домены у человека, без GO/source:
    res = search(
        query='P17931',# Search term for InterPro entry names or descriptions.
        # entry_type='family', #(str, optional): Filter by entry type (family, domain, etc.).
        # source_database='pfam', #(str, optional): Filter by member database (pfam, prosite, etc.).
        # go_term='GO:0006122',# (str, optional): Filter by GO term (e.g., "GO:0006122").
        species_filter="9606", #(str, optional): Filter by taxonomy ID (e.g., "9606" for human).
        page_size=1 #(int, optional): Number of results to return (max 200). Defaults to 20
    )
    hits = res.get("results", [])
    print(f"hits: {len(hits)}")
    for h in hits[:5]:
        acc = h["metadata"]["accession"]
        meta = get(accession=acc)
        print(acc, h["metadata"]["name"])
        # забери нужные поля из meta


hits: 0


In [115]:
res

{'error': 'Exception occurred: Expecting value: line 1 column 1 (char 0)'}